# Robustness dataset (HCP) - Schaefer100 topography


In [ ]:
from pathlib import Path
import sys
base_path = next(
    p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "CODE").is_dir() and (p / "DATA").is_dir()
)
sys.path.insert(0, str(Path(base_path) / 'CODE'))
from utils.paths import cluster_path, project_path


In [ ]:
import numpy as np
import pandas as pd
import os

## 1. Create and save dataframe

In [ ]:
hurst_path = f"{base_path}/DATA/robustness_dataset/Hurst_rest_all.mat"
atlas_info_path = f"{base_path}/DATA/templates/Schaefer2018_100Parcels_7Networks_order_FSLMNI152_2mm.Centroid_RAS.csv"

raw_output_path = f"{base_path}/RESULTS/robustness_dataset/hurst/dataframes/df_hurst_parcelwise_FIN_D4_modwt_1_5_padding_0_decimate_0_residuals_w_rest.csv"
cleaned_output_path = f"{base_path}/RESULTS/robustness_dataset/hurst/dataframes/df_hurst_parcelwise_FIN_D4_modwt_1_5_padding_0_decimate_0_residuals_w_rest_IQR_cleaned.csv"

num_parcels = 100

In [ ]:
from scipy.io import loadmat

hurst_data = loadmat(hurst_path)

H = hurst_data['H']
mean_H = np.mean(H, axis=1)
cortical_H = mean_H[:, :num_parcels] # shape: subject x parcel

sub_names = hurst_data['subjects'][0]

In [ ]:
parcel_info = pd.read_csv(atlas_info_path)

networks = parcel_info["ROI Name"].str.extract(
    r"^7Networks_[LR]H_([^_]+)"
)[0]

if len(networks) != num_parcels:
    raise ValueError(
        f"Expected {num_parcels} network labels, got {len(networks)}"
    )

if networks.isna().any():
    raise ValueError("Some network labels could not be extracted")

if cortical_H.shape != (len(sub_names), num_parcels):
    raise ValueError(
        f"Unexpected cortical_H shape: {cortical_H.shape}"
    )

# ------------------------------------------------------------
# Create long DataFrame
# ------------------------------------------------------------
df = pd.DataFrame({
    "Hursts": cortical_H.ravel(order="C"),
    "Subject": np.repeat(sub_names, num_parcels),
    "Parcel": np.tile(np.arange(1, num_parcels + 1), len(sub_names)),
    "Network": np.tile(networks.to_numpy(), len(sub_names)),
})

df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)

# ------------------------------------------------------------
# Save DataFrame
# ------------------------------------------------------------
os.makedirs(os.path.dirname(raw_output_path) or ".", exist_ok=True)
df.to_csv(raw_output_path, index=False)

print(f"Saved to: {raw_output_path}")
print(df.head())

## 2. Remove outliers

In [ ]:
# Function to remove outliers using IQR method
# outlier removal is done separately for each parcel and task combination

def remove_outliers_IQR_from_df(df, threshold=1.5, value_col="Hurst"):
    cleaned_groups = []
    for (_), group in df.groupby(["Parcel"]):
        data = group[value_col].values.astype(float)
        if len(data) <= 1:
            cleaned_groups.append(group)
            continue
        Q1, Q3 = np.percentile(data, [25, 75])
        IQR = Q3 - Q1
        if IQR == 0:
            cleaned_groups.append(group)
            continue
        lower_bound = Q1 - threshold * IQR
        upper_bound = Q3 + threshold * IQR
        mask = (data >= lower_bound) & (data <= upper_bound)
        cleaned_groups.append(group.loc[mask])
    cleaned_df = pd.concat(cleaned_groups, ignore_index=True)
    return cleaned_df

In [ ]:
# Load the DataFrame and prepare for analysis
hurst_df = pd.read_csv(raw_output_path)

parcels = np.unique(hurst_df['Parcel'])
networks = np.unique(hurst_df['Network'])

# Clean the DataFrame by removing outliers and save the cleaned version
cleaned_hurst_df = remove_outliers_IQR_from_df(hurst_df, threshold=1.5, value_col="Hursts")
cleaned_hurst_df.to_csv(cleaned_output_path, index=False)